# Sequence Model Training & Evaluation for IMU-based Exercise Classification

**Capstone -- Notebook 3 of 3.** Comparing models that classify which arm exercise is being performed from dual-IMU (forearm +
upper-arm) sensor data.

Structure: **Business Understanding -> Data Understanding -> Data Preparation -> Modeling -> Evaluation -> Findings -> Next Steps.**

## Business Understanding

**Research question:** *What is the best model for classifying which arm exercise is being performed from dual-IMU forearm +
upper-arm sensor data -- a classical model trained on aggregated per-window statistics (notebook `2.`), or a sequence model trained
directly on per-sample feature windows?*

**This notebook's role:** train and evaluate the sequence-model half of that comparison. Instead of collapsing each window into
mean/std/min/max/rms statistics the way notebook `2.` does, this notebook feeds the raw per-sample feature sequence for each window
into a small recurrent network (LSTM / GRU), which can in principle learn temporal shape that the aggregated statistics throw away
(e.g. *when* within a rep the elbow angle peaks, not just its min/max). Its held-out score is compared against notebook `2.`'s to
answer the capstone's research question: is that extra temporal detail worth the added data and compute cost?

## Imports

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score

import imu_dataset

plt.style.use("ggplot")
%matplotlib inline

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using device: {DEVICE}")

## Data Understanding

Unlike notebook `2.` (which reads the already-aggregated `data/exercise_windows.csv`), this notebook needs the raw per-sample
feature sequence *inside* each window, so it rebuilds the windowed dataset from `manifest.csv` via
`imu_dataset.build_sequence_dataset` -- the same windowing logic as notebook `1.` (`iter_windows` / `manifest.csv`), just without the
aggregation step. `X` is `(n_windows, seq_len, n_features)`, `y` is the per-window label array.

In [ ]:
WINDOW_SEC, HOP_SEC = 2.0, 1.0  # match notebook 1's windowing choice

X_raw, y_raw, meta = imu_dataset.build_sequence_dataset(
    "manifest.csv", repo_root=os.getcwd(), window_sec=WINDOW_SEC, hop_sec=HOP_SEC,
)
print(f"X: {X_raw.shape} (n_windows, seq_len, n_features)")
print(f"features: {imu_dataset._AGG_FIELDS}")
meta.head()

In [ ]:
class_counts = meta["label"].value_counts()
class_counts

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
class_counts.plot(kind="bar", ax=ax)
ax.set_ylabel("# windows"); ax.set_xlabel("exercise label"); ax.set_title("Windows per exercise label")
plt.tight_layout()

## Data Preparation

### Encode labels, scale features, and split

Same `READY_TO_MODEL` gate as notebook `2.` -- a stratified train/test split (and cross-validation within training) needs at least
two classes with at least a couple of windows each. Features are standardized per-channel (fit on the training set only, to avoid
leaking test-set statistics) since the LSTM's gradient-based training is sensitive to feature scale in a way tree-based models
aren't.

In [ ]:
MIN_CLASS_COUNT = class_counts.min()
N_CLASSES = class_counts.shape[0]
READY_TO_MODEL = N_CLASSES >= 2 and MIN_CLASS_COUNT >= 2

print(f"{N_CLASSES} class(es), smallest class has {MIN_CLASS_COUNT} window(s) -> READY_TO_MODEL = {READY_TO_MODEL}")

if READY_TO_MODEL:
    label_encoder = LabelEncoder().fit(y_raw)
    y_enc = label_encoder.transform(y_raw)

    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X_raw, y_enc, test_size=0.25, stratify=y_enc, random_state=RANDOM_STATE,
    )

    n_windows, seq_len, n_features = X_train_raw.shape
    scaler = StandardScaler().fit(X_train_raw.reshape(-1, n_features))
    X_train = scaler.transform(X_train_raw.reshape(-1, n_features)).reshape(X_train_raw.shape)
    X_test = scaler.transform(X_test_raw.reshape(-1, n_features)).reshape(X_test_raw.shape)

    print(f"train: {X_train.shape}, test: {X_test.shape}, classes: {list(label_encoder.classes_)}")
else:
    print(
        "Not enough labeled classes yet to hold out a stratified test set. "
        "See notebook 1's Findings section -- record and label at least one more exercise, "
        "rerun notebook 1, then rerun this notebook."
    )

## Modeling

### Model definition

Two recurrent architectures are compared as part of the hyperparameter search below -- a single-layer **LSTM** and a single-layer
**GRU** (GRU has fewer parameters, which can matter with a small dataset) -- each followed by a linear classification head on the
final hidden state.

In [ ]:
class SequenceClassifier(nn.Module):
    def __init__(self, n_features, hidden_size, n_classes, cell_type="lstm"):
        super().__init__()
        rnn_cls = {"lstm": nn.LSTM, "gru": nn.GRU}[cell_type]
        self.rnn = rnn_cls(input_size=n_features, hidden_size=hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, n_classes)

    def forward(self, x):
        _, hidden = self.rnn(x)
        h_n = hidden[0] if isinstance(hidden, tuple) else hidden  # LSTM returns (h_n, c_n), GRU returns h_n
        return self.head(h_n[-1])


def train_one_model(cell_type, hidden_size, lr, X_tr, y_tr, X_val, y_val, n_classes, n_epochs=30):
    model = SequenceClassifier(X_tr.shape[-1], hidden_size, n_classes, cell_type=cell_type).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_tr).float(), torch.from_numpy(y_tr).long()),
        batch_size=8, shuffle=True,
    )
    for _ in range(n_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss_fn(model(xb), yb).backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        val_pred = model(torch.from_numpy(X_val).float().to(DEVICE)).argmax(dim=1).cpu().numpy()
    return model, f1_score(y_val, val_pred, average="macro", zero_division=0)

### Cross-validated grid search

`GridSearchCV` doesn't wrap a PyTorch training loop directly, so the grid search here is a small manual loop over
(`cell_type`, `hidden_size`, `learning_rate`) combinations, each scored with the same stratified k-fold cross-validation notebook
`2.` uses, on the same `f1_macro` metric (see Evaluation below for the rationale).

In [ ]:
if READY_TO_MODEL:
    param_grid = [
        {"cell_type": cell_type, "hidden_size": hidden_size, "lr": lr}
        for cell_type in ("lstm", "gru")
        for hidden_size in (16, 32)
        for lr in (1e-2, 1e-3)
    ]

    n_splits = min(3, MIN_CLASS_COUNT)  # can't have more folds than the smallest class
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    grid_results = []
    for params in param_grid:
        fold_scores = []
        for train_idx, val_idx in cv.split(X_train, y_train):
            _, val_f1 = train_one_model(
                params["cell_type"], params["hidden_size"], params["lr"],
                X_train[train_idx], y_train[train_idx], X_train[val_idx], y_train[val_idx],
                n_classes=N_CLASSES,
            )
            fold_scores.append(val_f1)
        grid_results.append({**params, "cv_f1_macro_mean": np.mean(fold_scores), "cv_f1_macro_std": np.std(fold_scores)})

    grid_results_df = pd.DataFrame(grid_results).sort_values("cv_f1_macro_mean", ascending=False)
    best_params = grid_results_df.iloc[0][["cell_type", "hidden_size", "lr"]].to_dict()
    print("best hyperparameters:", best_params)
    grid_results_df.head()
else:
    print("skipped -- see Data Preparation")

## Evaluation

**Evaluation metric: macro-averaged F1** -- the same metric and the same rationale as notebook `2.`, so the two notebooks' held-out
scores are directly comparable: every exercise class matters equally regardless of how often it appears, which plain accuracy
doesn't guarantee once classes are imbalanced across subjects/sessions.

The best hyperparameters from the grid search are retrained on the *full* training set (not just one fold) and evaluated once on
the held-out test set, which was untouched by cross-validation or grid search.

In [ ]:
if READY_TO_MODEL:
    final_model, _ = train_one_model(
        best_params["cell_type"], int(best_params["hidden_size"]), best_params["lr"],
        X_train, y_train, X_test, y_test, n_classes=N_CLASSES, n_epochs=50,
    )
    final_model.eval()
    with torch.no_grad():
        y_pred = final_model(torch.from_numpy(X_test).float().to(DEVICE)).argmax(dim=1).cpu().numpy()

    test_f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)
    print(f"held-out test macro-F1: {test_f1_macro:.3f}")
    print(classification_report(
        y_test, y_pred, labels=range(N_CLASSES), target_names=label_encoder.classes_, zero_division=0,
    ))
else:
    print("skipped -- see Data Preparation")

In [ ]:
if READY_TO_MODEL:
    labels_order = list(label_encoder.classes_)
    cm = confusion_matrix(y_test, y_pred, labels=range(N_CLASSES))
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels_order, yticklabels=labels_order, ax=ax)
    ax.set_xlabel("predicted label"); ax.set_ylabel("true label"); ax.set_title(f"Confusion matrix -- tuned {best_params['cell_type'].upper()}")
    plt.tight_layout()
else:
    print("skipped -- see Data Preparation")

## Findings

*Template -- fill in once `READY_TO_MODEL` is `True` and the cells above have real results.*

- **Best model:** *(name the winning architecture/hyperparameters and its held-out macro-F1 score in plain language.)*
- **Notebook `2.` vs. notebook `3.`:** *(state which approach won and by how much -- e.g. "the sequence model scored X macro-F1
  versus the classical model's Y, a difference of Z" -- and give the plain-language answer to the capstone's research question.)*
- **Where it struggles:** *(read off the confusion matrix, as in notebook `2.`.)*
- **Actionable item:** *(if the classical model matched or beat the sequence model, the recommendation is to ship the classical
  model -- it's cheaper to train and easier to interpret for the same or better accuracy. If the sequence model won by a
  meaningful margin, the recommendation is to invest in recording more data, since sequence models need more of it than classical
  ones to generalize well.)*

## Next Steps and Recommendations

1. Once notebook `1.`'s dataset has more than one exercise class, rerun this notebook top to bottom -- `READY_TO_MODEL` will flip to
   `True` and every modeling/evaluation cell above will run for real with no code changes.
2. Compare this notebook's held-out macro-F1 directly against notebook `2.`'s to answer the capstone's research question.
3. If the dataset stays small (tens of windows per class) even after more recording, expect the classical model in notebook `2.`
   to be competitive or better -- sequence models like this LSTM/GRU generally need substantially more data than that to reliably
   outperform aggregated-statistics baselines.
4. If the sequence model is worth pursuing further, the next architecture improvement to try is a second stacked recurrent layer
   or an attention pooling layer over the hidden states, rather than only the final hidden state used here.